# Preprocessing testing

In [7]:
import dask.dataframe as dd
import os
import pandas as pd
import xarray as xr
from jetstream_interpolate_convcnp.utils.constants import ALTITUDE, GEOPOTENTIAL_HEIGHT, LONGITUDE, LATITUDE, PRESSURE_LEVEL, TIME, WIND_U, WIND_V
import re
import numpy as np

In [3]:
# ecmwf to offgrid parquet

ds_in_path = "/mnt/hdd/jetstream/data/development/ecmwf/july_2019/ecmwf_forecast.nc"
ds = xr.open_dataset(ds_in_path)

level_var_pattern = re.compile(r"^(UGRD|VGRD|HGT)_(\d+)mb$")
target_name_by_source = {
    "UGRD": WIND_U,
    "VGRD": WIND_V,
    "HGT": ALTITUDE,
}

grouped_vars = {WIND_U: [], WIND_V: [], ALTITUDE: []}

for var_name in list(ds.data_vars):
    match = level_var_pattern.match(var_name)
    if not match:
        continue

    source_name, level = match.groups()
    grouped_vars[target_name_by_source[source_name]].append((int(level), var_name))

vars_to_drop = []
for target_name, level_vars in grouped_vars.items():
    if not level_vars:
        continue

    level_vars = sorted(level_vars, key=lambda item: item[0])
    arrays = [
        ds[var_name].expand_dims({PRESSURE_LEVEL: [level]})
        for level, var_name in level_vars
    ]
    ds[target_name] = xr.concat(arrays, dim=PRESSURE_LEVEL)
    vars_to_drop.extend([var_name for _, var_name in level_vars])

if vars_to_drop:
    ds = ds.drop_vars(vars_to_drop)

rename_map = {}
if "latitude" in ds.coords and LATITUDE not in ds.coords:
    rename_map["latitude"] = LATITUDE
if "longitude" in ds.coords and LONGITUDE not in ds.coords:
    rename_map["longitude"] = LONGITUDE
if "time" in ds.coords and TIME != "time" and TIME not in ds.coords:
    rename_map["time"] = TIME

if rename_map:
    ds = ds.rename(rename_map)

In [4]:
df = ds.to_dask_dataframe().reset_index()

In [16]:
df.head()

,index,lat,lon,time,pressure_level,u,v,altitude,altitude_band,log_altitude,coarse_lat,coarse_lon
0,0,-90.0,0.0,2019-07-01,200,2.644302,-1.617889,10574.985352,5.0,9.266247,-90.0,0.0
1,1,-90.0,0.0,2019-07-01,250,6.087719,2.137413,9289.524414,4.0,9.136642,-90.0,0.0
2,2,-90.0,0.0,2019-07-01,300,6.933186,2.614273,8206.794922,4.0,9.012718,-90.0,0.0
3,3,-90.0,0.0,2019-07-01,500,1.535641,-1.159138,4898.779785,2.0,8.496741,-90.0,0.0
4,4,-90.0,0.0,2019-07-01,700,-2.965982,3.703644,2585.971191,1.0,7.857856,-90.0,0.0
